# Multiobjective Evolutionary Algorithm based on Decomposition
-- Uvini Balasuriya and Divya Ramachandran

All the imports

In [1]:
import pandas as pd
import numpy as np
from pymoo.problems import get_problem

Function for implementing moea-d

In [ ]:
# Function to evaluate the problem
def evaluate(problem, x):
    """Evaluate the problem for a given input vector.
    
    problem: The problem instance from pymoo
    x: The input vector to evaluate
    """
    return problem.evaluate(x.reshape(1, -1))[0]


# Function to generate weights vectors
# MOEA-D relies on evenly distributed weight vectors to decompose the multi-objective problem into scalar subproblems.

def generate_weights(N, M):

    """
    Generate weight vectors for MOEA-D.
    N: Number of weight vectors (population size)
    M: Number of objectives
    """

    if M == 2:
        return np.array([[i/(N-1), 1 - i/(N-1)] for i in range(N)])
    else:
        # fallback random for higher dimensions
        W = np.random.rand(N, M)
        return W / W.sum(axis=1, keepdims=True)
    
    
# Neighbourhood function to find the closest weight vectors
def find_neighbors(W, T):
    """
    Find the T closest weight vectors for each weight vector in W.
    W: Weight vectors
    T: Number of neighbors
    """
    from sklearn.metrics import pairwise_distances
    distances = pairwise_distances(W)
    neighbors = np.argsort(distances, axis=1)[:, :T]
    return neighbors


# Iniitalize population
def initialize_population(problem, N):
    """
    Initialize the population for MOEA-D.
    """
    X = np.random.uniform(problem.xl, problem.xu, (N, problem.n_var))
    F = problem.evaluate(X)
    return X, F


# Initialize ideal point Z*
def initialize_ideal_point(F):
    """
    Initialize the ideal point Z* based on the initial population's objective values.
    F: Objective values of the initial population
    """
    return np.min(F, axis=0)


# Decomposition function to solve scalar subproblems
def g(f, w, z):
    """
    Tchebycheff decomposition function.

    f: objective vector of ONE solution (shape: M,)
    w: ONE weight vector (shape: M,)
    z: ideal point (shape: M,)

    returns: scalar value
    """
    return np.max(w * np.abs(f - z))


# Function to select parents for reproduction
def select_parents(i, B, N, delta):
    """
    Select parents for reproduction based on the neighborhood structure.
        i: Index of the current subproblem
        B: Neighborhood structure (list of neighbors for each subproblem)
        N: Total population size
        delta: Probability of selecting from the neighborhood
    """

    if np.random.rand() < delta:
        P = B[i]            # neighbors
    else:
        P = np.arange(N)    # whole population

    k, l = np.random.choice(P, 2, replace=False)
    return k, l


def variation(x1, x2, xl, xu):
    """
    x1, x2: parent solutions (1D arrays)
    xl, xu: lower/upper bounds (from problem)

    returns: offspring y
    """

    # Crossover (blend parents)
    alpha = np.random.rand()
    y = alpha * x1 + (1 - alpha) * x2

    # Mutation (add small noise)
    sigma = 0.1  # mutation strength (tune if needed)
    noise = sigma * (xu - xl) * np.random.randn(len(x1))
    y = y + noise

    # Clip to bounds
    y = np.clip(y, xl, xu)

    return y

